In [ ]:
"""
VaLiK (ICCV 2025) -- reduced-run ADAPTATION to GQA visual-CoT val.  [v4, standalone]

STANDALONE AND REPRODUCIBLE. Nothing is read from a previous run: no resume, no
cached graphs, no re-parsing. Given the two mounts and the seed, this notebook
regenerates every artefact it reports on. Artefacts are still WRITTEN, for audit,
but never read back.

Reproducibility contract:
  - python, numpy and torch are seeded; all decoding is greedy (do_sample=False).
  - The image subsample is sorted-then-seeded, so MAX_IMAGES=N always selects the
    same N images.
  - Remaining nondeterminism: cuDNN/attention kernel selection, fp16 reduction
    order, and batch composition (padding changes fp16 arithmetic slightly).
    Expect exact-match to move by a fraction of a point between runs, not more.
  - Model weights are pinned by revision in CONFIG.

Carried over from the v1-v3 debugging, all implementation bugs in this file and
none of them properties of the paper:
  - BLIP-2 is captioned unconditionally (Eq. 2 sets S_0 := empty); v1 passed a
    prompt and BLIP-2-OPT echoed it into the output.
  - A fully pruned S_hat yields an EMPTY subgraph and is never sent to the
    extractor; v1 called the LLM on an empty string and it invented a graph.
  - Extraction JSON is salvaged object-by-object, so a generation that hits the
    token cap costs the last item instead of the whole response. In the v2 run
    353/1200 and 206/1130 generations were capped, and strict json.loads threw
    away every complete object alongside the broken one.
  - The leakage guard asserts on `reasoning`/`thought`, not `full_answer`. GQA's
    full_answer is a short template ("the bus is on the street."), which a correct
    description reproduces by construction; asserting on it flagged success as
    fraud and killed a finished run.

Section order follows the requested spec:
  CONFIG -> LOAD DATA -> PREPROCESSING -> FEATURE SELECTION -> MODEL -> TRAIN
  -> EVALUATE -> SAVE RESULTS
"""


In [ ]:
# %% [CELL 0] ----------------------------------------------------------------
# INSTALL + IMPORTS + SEEDS
# ----------------------------------------------------------------------------
# Pinned. Kaggle's torch build is left alone on purpose (T4 = sm75, fp16 only).
import subprocess, sys

PIP_PINS = [
    "transformers==4.51.3",
    "accelerate==1.6.0",
    "tokenizers==0.21.1",
    "safetensors==0.5.3",
    "sentencepiece==0.2.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PINS], check=True)

import gc
import json
import os
import random
import re
import string
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# NONDETERMINISM THAT REMAINS: cuDNN/attention kernel selection, fp16 reduction
# order, and batch composition (padding changes fp16 arithmetic slightly). All
# decoding is greedy (do_sample=False), so there is no sampling randomness.
torch.backends.cudnn.benchmark = True

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| n_gpu", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name} {p.total_memory/1e9:.1f}GB sm{p.major}{p.minor}")


In [ ]:
# %% [CELL 1] ----------------------------------------------------------------
# ===== CONFIG =====
# ----------------------------------------------------------------------------
@dataclass
class CFG:
    # --- paths (both mount points, as given) ---
    EVAL_JSONL: str = ("/kaggle/input/test-dataset-visual-cot/visual-cot/"
                       "cot_with_detailed_reasoning_steps/gqa_cot_val.jsonl")
    IMAGE_ROOT: str = "/kaggle/input/gqa-images/images"
    OUT_DIR: str = "/kaggle/working"

    # --- expected shape of the val file (used only for reporting) ---
    EXPECTED_RECORDS: int = 9855
    EXPECTED_IMAGES: int = 5422

    # --- reduced run size. DEVIATION: paper scores the full benchmark. ---
    MAX_IMAGES: int = 800           # ~1.82 records/image -> ~1450 records
    SEED: int = 0

    # --- CoE cascade (Sec. 3.1). DEVIATION: paper chains BLIP-2 + LLaVA +
    #     Qwen2-VL (Sec. 4.1); LLaVA-7B+ does not co-fit here in the time budget.
    VLM_CHAIN: tuple = ("blip2", "qwen2vl")
    BLIP2_ID: str = "Salesforce/blip2-opt-2.7b"   # ASSUMPTION #6
    QWEN2VL_ID: str = "Qwen/Qwen2-VL-2B-Instruct"
    CLIP_ID: str = "openai/clip-vit-large-patch14"  # Sec. 4.1: CLIP-ViT-L/14
    LLM_ID: str = "Qwen/Qwen2.5-7B-Instruct"      # Sec. 4.1 base reasoning model
    KG_LLM_ID: str = "Qwen/Qwen2.5-7B-Instruct"   # DEVIATION: paper = DeepSeek-R1-70B

    # ASSUMPTION: paper does not specify C in Eq. 6; using 1 pass over the chain.
    COE_ITERATIONS: int = 1

    # Sec. 3.2 / Sec. 4.1. ASSUMPTION #2: paper gives 0.20 (ScienceQA) and 0.25
    # (CrisisMMD); GQA is a QA benchmark so 0.20 is used.
    TAU: float = 0.20
    CLIP_TEXT_MAXLEN: int = 77      # ASSUMPTION #10: CLIP text encoder limit

    # ASSUMPTION #3: paper does not specify how many triplets are retrieved.
    TOP_K_TRIPLETS: int = 15

    # ASSUMPTION #9: paper does not specify generation lengths or batch sizes.
    BLIP2_MAX_NEW: int = 48
    QWEN2VL_MAX_NEW: int = 256   # v1 used 128 and truncated 1056/1200 mid-sentence
    KG_MAX_NEW: int = 512        # v1 used 384
    QA_MAX_NEW: int = 24
    BLIP2_BATCH: int = 8
    QWEN2VL_BATCH: int = 4
    KG_BATCH: int = 8
    QA_BATCH: int = 8
    LLM_MAX_INPUT_TOKENS: int = 1536
    QWEN2VL_MAX_PIXELS: int = 640 * 28 * 28

    # ASSUMPTION #7: "full frame" is not defined by the paper; this is the
    # requested sanity split, not a paper quantity.
    FULL_FRAME_AREA_FRAC: float = 0.80

    # ASSUMPTION #4 (revised): the paper does not give the extraction prompt or
    # its serialization. v1 assumed LightRAG-style delimited tuples and parsed
    # 0.8%/8.2% of responses. "json" and "tuple" are both available; run
    # probe_kg_extraction.py to see which one this model actually follows.
    KG_FORMAT: str = "json"
    PROBE_N: int = 8             # images extracted before the full pass
    PROBE_MIN_PARSE_RATE: float = 0.5   # below this the run stops, not warns

    # Eq. 9 builds G from S_hat. If S_hat is empty there is nothing to build.
    SKIP_EMPTY_SHAT: bool = True
    DROP_TRUNCATED_TAIL: bool = True

    # Omit the "description" field from the extraction schema. Retrieval (Eq. 11)
    # matches on h/r/t only, and in the v2 run the model left this field empty for
    # most items anyway, so it was pure decode cost and a truncation risk.
    KG_EMIT_DESCRIPTIONS: bool = False

    # Off by default. The v2 graphs contain off-spec triplets: self-loops encoding
    # attributes ("tiles -> are large -> tiles") and verbs in the target slot
    # ("man -> wearing"). Eq. 10 defines triplets over entity sets, so these are
    # malformed, but dropping them is a cleaning step the paper does not describe.
    # Turning this on is a deliberate deviation, not a fix.
    DROP_MALFORMED_TRIPLETS: bool = False

    # The run stops before the expensive passes if the projection exceeds this.
    MAX_RUNTIME_HOURS: float = 9.0

    # Ablation rows: Table 5 structure (base / +CVs / +SV).
    RUN_ABLATION: bool = True

    # Guards used by EVALUATE. Not paper quantities and not used in any metric:
    # LEAK_MIN_CHARS is the shortest answer-derived field worth substring-testing
    # (short strings collide by chance); EMPTY_PRED_WARN_DIV=10 warns when more
    # than 1/10 of predictions normalize to the empty string.
    LEAK_MIN_CHARS: int = 12
    EMPTY_PRED_WARN_DIV: int = 10

    CALIB_N: int = 16               # images used to print a wall-clock projection

    def __post_init__(self):
        os.makedirs(self.OUT_DIR, exist_ok=True)


cfg = CFG()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32  # T4: fp16, no bf16
# torch.amp autocast is not used: weights are loaded in fp16 and nothing is
# trained, so there is no fp32 master copy for autocast to manage. On CPU the
# code falls back to fp32 and still runs, just slowly.

# Write-only. Nothing in this notebook reads these back.
CAPTIONS_PATH = os.path.join(cfg.OUT_DIR, "captions.json")
PRUNED_PATH = os.path.join(cfg.OUT_DIR, "pruned_descriptions.json")
KG_RAW_PATH = os.path.join(cfg.OUT_DIR, "kg_cvs.json")   # from unpruned S (+CVs)
KG_SV_PATH = os.path.join(cfg.OUT_DIR, "kg_sv.json")     # from pruned  S-hat (+SV)
RAW_KG_LOG = os.path.join(cfg.OUT_DIR, "kg_raw_outputs.jsonl")


def print_deviations():
    rows = [
        ("Benchmark", "CrisisMMD / ScienceQA (Sec. 4.1)", "GQA visual-CoT val",
         "different task; not comparable to Tables 1-5"),
        ("Metric", "accuracy percentage (Sec. 4.1)", "normalized exact match",
         "open-ended scoring is stricter than multiple choice"),
        ("GPU", "1xA100-80GB (Sec. 4.1)", f"{torch.cuda.device_count()}xT4 16GB fp16",
         "forces all model substitutions below"),
        ("Graph LLM", "DeepSeek-R1-70B (Sec. 4.1)", cfg.KG_LLM_ID,
         "weaker extraction -> sparser/noisier triplets"),
        ("VLM chain", "BLIP-2 + LLaVA + Qwen2-VL (Sec. 4.1)", str(cfg.VLM_CHAIN),
         "Fig. 4: fewer VLMs -> smaller CVs gain"),
        ("VLM size", "LLaVA-34B / Qwen2-VL-72B-I (Table 2)", "2.7B / 2B",
         "shorter, less detailed descriptions"),
        ("Retrieval", "LightRAG hybrid (Sec. 4.1)", "keyword dual-level over subgraph",
         "different retrieval algorithm"),
        ("KG scope", "entire training set as KB (Sec. 4.1)", "per-image subgraphs of eval images",
         "GQA questions are image-specific; no labels used"),
        ("Configs", "Image-only and Text-Image (Sec. 4.1)", "Image-only only",
         "GQA text fields are answer-derived -> leakage"),
        ("tau", "0.20 ScienceQA / 0.25 CrisisMMD (Sec. 4.1)", str(cfg.TAU),
         "assumption for an unseen benchmark"),
        ("Records", f"{cfg.EXPECTED_RECORDS} val records", f"~1.82 x {cfg.MAX_IMAGES}",
         "SUBSET SCORE -- not the val score"),
    ]
    print("\n=== DEVIATIONS FROM THE PAPER ===")
    print(pd.DataFrame(rows, columns=["item", "paper", "here", "effect"]).to_string(index=False))
    print("=================================\n")


print_deviations()
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=2))


# ===== LOAD DATA =====
def hard_fail(msg):
    """No fallbacks anywhere in this notebook. A broken path stops the run."""
    raise RuntimeError(msg)


if not os.path.exists(cfg.EVAL_JSONL):
    print("--- what is actually mounted under /kaggle/input ---")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - 2
        if depth <= 3:
            print("  " * depth + os.path.basename(root) + "/  ({} files)".format(len(files)))
    hard_fail(f"eval jsonl not found: {cfg.EVAL_JSONL}")

assert cfg.EVAL_JSONL.endswith("gqa_cot_val.jsonl"), \
    "scoring must use the val file; the train file is off limits"

records = []
with open(cfg.EVAL_JSONL) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"loaded {len(records)} records from {cfg.EVAL_JSONL}")
if len(records) != cfg.EXPECTED_RECORDS:
    print(f"  NOTE: expected {cfg.EXPECTED_RECORDS} records, found {len(records)}")

REQUIRED_FIELDS = ["question", "answer", "full_answer", "image", "width", "height",
                   "bboxs", "dataset", "split", "reasoning", "thought"]
missing = [k for k in REQUIRED_FIELDS if k not in records[0]]
if missing:
    hard_fail(f"record schema mismatch, missing fields: {missing}")

# Fields that are derived from the ground truth answer. These are NEVER read by
# the construction or inference code. Only `question` (the query q, Eq. 11) and
# `image` are consumed; `answer` is consumed only by the scorer.
FORBIDDEN_FIELDS = ["answer", "full_answer", "reasoning", "thought"]

print("split value counts:", pd.Series([r["split"] for r in records]).value_counts().to_dict())
print("dataset value counts:", pd.Series([r["dataset"] for r in records]).value_counts().to_dict())


# ===== SETUP CHECKS (the "cell 1" checks) =====
if not os.path.isdir(cfg.IMAGE_ROOT):
    print("--- what is actually mounted under /kaggle/input ---")
    for name in sorted(os.listdir("/kaggle/input")):
        print("  ", name)
    hard_fail(f"IMAGE_ROOT is not a directory: {cfg.IMAGE_ROOT}")

n_files = len(os.listdir(cfg.IMAGE_ROOT))
print(f"\nMOUNT 1 (annotations): {cfg.EVAL_JSONL}")
print(f"MOUNT 2 (images)     : {cfg.IMAGE_ROOT}  -> {n_files} files")

unique_images = sorted({r["image"] for r in records})
print(f"unique images referenced by val: {len(unique_images)} "
      f"(expected {cfg.EXPECTED_IMAGES})")
print(f"records per image: {len(records)/max(len(unique_images),1):.2f}")

missing_imgs = [im for im in unique_images
                if not os.path.exists(os.path.join(cfg.IMAGE_ROOT, im))]
print(f"resolved: {len(unique_images)-len(missing_imgs)}/{len(unique_images)}")
if missing_imgs:
    print("first 20 missing filenames:")
    for m in missing_imgs[:20]:
        print("   ", m)
    hard_fail(f"{len(missing_imgs)} images do not resolve under IMAGE_ROOT. "
              "Fix the path. Substitute images are never generated.")

# Real size vs declared width/height. bboxs are pixel coordinates, so a resized
# or re-encoded image puts every box in the wrong place.
real_size = {}
for im in tqdm(unique_images, desc="reading image headers"):
    with Image.open(os.path.join(cfg.IMAGE_ROOT, im)) as pil:
        real_size[im] = pil.size  # (w, h), header only

size_ok, size_bad = [], []
for r in records:
    w, h = real_size[r["image"]]
    (size_ok if (w == int(r["width"]) and h == int(r["height"])) else size_bad).append(r)

print(f"\nsize check: kept {len(size_ok)}, DROPPED {len(size_bad)} "
      f"({100*len(size_bad)/len(records):.2f}%) for width/height disagreement")
if size_bad:
    ex = size_bad[0]
    print(f"  example: {ex['image']} declared {ex['width']}x{ex['height']} "
          f"actual {real_size[ex['image']][0]}x{real_size[ex['image']][1]}")
if len(size_bad) > 0.05 * len(records):
    print("  WARNING: >5% mismatch. The images were likely re-encoded and the "
          "boxes cannot be trusted. Box-derived splits below are unreliable.")

records = size_ok

# Clamp boxes to the frame.
n_clamped = 0
for r in records:
    w, h = float(r["width"]), float(r["height"])
    new_boxes = []
    for b in r["bboxs"]:
        x1, y1, x2, y2 = [float(v) for v in b]
        cx1, cy1 = min(max(x1, 0.0), w), min(max(y1, 0.0), h)
        cx2, cy2 = min(max(x2, 0.0), w), min(max(y2, 0.0), h)
        if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
            n_clamped += 1
        new_boxes.append([cx1, cy1, cx2, cy2])
    r["bboxs_clamped"] = new_boxes
print(f"clamped {n_clamped} boxes to [0,width] x [0,height]")

# Full-frame flag. ASSUMPTION #7: threshold is not a paper quantity.
for r in records:
    area = float(r["width"]) * float(r["height"])
    frac = 0.0
    for (x1, y1, x2, y2) in r["bboxs_clamped"]:
        bw, bh = max(0.0, x2 - x1), max(0.0, y2 - y1)
        frac = max(frac, (bw * bh) / area if area > 0 else 0.0)
    r["max_box_frac"] = frac
    r["is_full_frame"] = frac >= cfg.FULL_FRAME_AREA_FRAC
n_full = sum(r["is_full_frame"] for r in records)
print(f"full-frame records (max box area >= {cfg.FULL_FRAME_AREA_FRAC:.2f} of image): "
      f"{n_full} / {len(records)}")

# GROUNDING ACCURACY: N/A. VaLiK emits no bounding boxes anywhere in Sec. 3;
# entity-to-image linkage is image-level (Sec. 3.3), not region-level. There is
# no predicted box, so there is no IoU to compute and no grounding accuracy to
# report. The full-frame flag is instead used to report exact-match twice --
# with and without those records -- in the EVALUATE section.
print("grounding accuracy: N/A (the paper's method predicts no boxes; see Sec. 3.3)")


In [ ]:
# %% [CELL 2] ----------------------------------------------------------------
# ===== PREPROCESSING =====
# ----------------------------------------------------------------------------
rng = random.Random(cfg.SEED)
all_imgs = sorted({r["image"] for r in records})
if cfg.MAX_IMAGES >= len(all_imgs):
    sel_images = all_imgs
    print(f"using all {len(sel_images)} images")
else:
    sel_images = sorted(rng.sample(all_imgs, cfg.MAX_IMAGES))
    print(f"DEVIATION: sampled {len(sel_images)} of {len(all_imgs)} images "
          f"(seed={cfg.SEED}). This is a SUBSET SCORE.")

sel_set = set(sel_images)
eval_records = [r for r in records if r["image"] in sel_set]
for i, r in enumerate(eval_records):
    r["rid"] = i
print(f"eval records: {len(eval_records)} over {len(sel_images)} images")

# Zero-shot: nothing is fit on any split (Sec. 1, "zero-shot"). Assert it.
TRAIN_RECORD_IDS = set()          # deliberately empty
EVAL_RECORD_IDS = {r["rid"] for r in eval_records}
assert TRAIN_RECORD_IDS.isdisjoint(EVAL_RECORD_IDS), "train/eval index overlap"
assert len(TRAIN_RECORD_IDS) == 0, (
    "This pipeline trains nothing and fits no scaler/encoder/selector. "
    "There is no train split to leak from.")
print("leakage guard: no training split exists; no estimator is fit on labels.")

ans_series = pd.Series([r["answer"] for r in eval_records])
print(f"\nanswer 'class' distribution (eval split): {ans_series.nunique()} unique strings")
print(ans_series.value_counts().head(20).to_string())
print(f"singleton answers: {(ans_series.value_counts()==1).sum()}")


def load_rgb(fname):
    with Image.open(os.path.join(cfg.IMAGE_ROOT, fname)) as im:
        return im.convert("RGB").copy()


In [ ]:
# %% [CELL 3] ----------------------------------------------------------------
# ===== FEATURE SELECTION -- Cross-Modal Similarity Verification (Sec. 3.2) =====
# ----------------------------------------------------------------------------
# Sec. 3.2: windows W_k are scored by CLIP cosine (Eq. 7) and any window with
# alpha_k < tau is discarded; S-hat is the union of the survivors (Eq. 8).
# Sec. 3.2 states the window size m "adapts dynamically to natural sentence
# segmentation", so windows here are sentences.
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")


def split_windows(text):
    """Sentence-level windows W_k (Sec. 3.2: dynamic, sentence-adaptive)."""
    parts = [p.strip() for p in _SENT_SPLIT.split(text.strip()) if p.strip()]
    return parts


class SimilarityVerifier:
    """Eq. 7-8 with a frozen CLIP-ViT-L/14 encoder (Sec. 3.2, Sec. 4.1)."""

    def __init__(self, model, processor, tau):
        self.model, self.proc, self.tau = model, processor, tau

    @torch.no_grad()
    def scores(self, image, windows):
        if not windows:
            return []
        inputs = self.proc(text=windows, images=image, return_tensors="pt",
                           padding=True, truncation=True,
                           max_length=cfg.CLIP_TEXT_MAXLEN)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        img_emb = self.model.get_image_features(pixel_values=inputs["pixel_values"])
        txt_emb = self.model.get_text_features(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
        # Eq. 7: raw cosine. The paper states alpha_k lies in [0,1] (Sec. 3.2);
        # CLIP cosines can dip below 0, and no clipping is described, so none is
        # applied.
        return (txt_emb @ img_emb.T).squeeze(-1).float().cpu().tolist()

    def prune(self, image, text):
        """Returns (S_hat, kept_windows, all_alphas). Eq. 8."""
        wins = split_windows(text)
        alphas = self.scores(image, wins)
        kept = [w for w, a in zip(wins, alphas) if a >= self.tau]
        return " ".join(kept), kept, alphas


In [ ]:
# %% [CELL 4] ----------------------------------------------------------------
# ===== MODEL =====
# ----------------------------------------------------------------------------
from transformers import (AutoModelForCausalLM, AutoProcessor, AutoTokenizer,
                          Blip2ForConditionalGeneration, CLIPModel, CLIPProcessor,
                          Qwen2VLForConditionalGeneration)

PARAM_COUNTS = {}      # model id -> parameter count, measured at load time
STAGE_SECONDS = {}     # stage name -> wall clock seconds, measured


def _count(model, key):
    n = sum(p.numel() for p in model.parameters())
    PARAM_COUNTS[key] = n
    print(f"  {key}: {n/1e9:.3f}B parameters")
    return n


def _max_memory_map(headroom_gb=1.6):
    if DEVICE != "cuda":
        return None
    mm = {}
    for i in range(torch.cuda.device_count()):
        tot = torch.cuda.get_device_properties(i).total_memory / 1e9
        mm[i] = f"{max(tot - headroom_gb, 1.0):.1f}GiB"
    mm["cpu"] = "24GiB"     # accelerate offloads here only if the GPUs are full
    return mm


def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def peak_gpu_gb():
    if DEVICE != "cuda":
        return float("nan")
    return sum(torch.cuda.max_memory_allocated(i)
               for i in range(torch.cuda.device_count())) / 1e9


def strip_prompt_echo(text):
    """BLIP-2-OPT prepends its own prompt to the decoded string."""
    t = text.strip()
    t = re.sub(r"^a detailed photo description\s*:?\s*", "", t, flags=re.I)
    # Coordinate soup like "(13,15),(987,985)" is not a description.
    if re.fullmatch(r"[\s\d(),.\-]*", t):
        return ""
    return t


def strip_incomplete_tail(text):
    """Drop a trailing sentence fragment left by the token cap."""
    if not cfg.DROP_TRUNCATED_TAIL:
        return text
    t = text.strip()
    if not t or t[-1] in ".!?":
        return t
    cut = max(t.rfind("."), t.rfind("!"), t.rfind("?"))
    return t[:cut + 1].strip() if cut > 0 else t


# --- Expert 1: BLIP-2 (Sec. 4.1: "a chain of VLMs including BLIP-2, LLaVA, and
#     Qwen2-VL"; Sec. 4.4 adopts BLIP-2 as the primary model) -----------------
class Blip2Expert:
    """E_1 in Eq. 2. S_0 := empty set (Sec. 3.1), so this expert sees only I."""
    name = "blip2"

    def __init__(self):
        self.proc = AutoProcessor.from_pretrained(cfg.BLIP2_ID)
        self.proc.tokenizer.padding_side = "left"
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            cfg.BLIP2_ID, torch_dtype=DTYPE)
        self.model.to(DEVICE if DEVICE == "cpu" else "cuda:0").eval()
        _count(self.model, cfg.BLIP2_ID)

    @torch.no_grad()
    def describe(self, images, prev_texts):
        # prev_texts is ignored for expert 1 by construction (S_0 = empty, Sec. 3.1).
        # v1 passed a text prompt here; BLIP-2-OPT echoes its prompt into the
        # decoded output, which is where "a detailed photo description:(13,15),..."
        # came from. Unconditional captioning is both cleaner and closer to
        # S_0 := empty.
        inputs = self.proc(images=images, return_tensors="pt")
        inputs = {k: (v.to(self.model.device, DTYPE) if v.dtype.is_floating_point
                      else v.to(self.model.device)) for k, v in inputs.items()}
        out = self.model.generate(**inputs, max_new_tokens=cfg.BLIP2_MAX_NEW,
                                  do_sample=False, num_beams=1)
        txt = self.proc.batch_decode(out, skip_special_tokens=True)
        return [strip_prompt_echo(t) for t in txt]


# --- Expert 2: Qwen2-VL (Sec. 4.1). Eq. 2: E_i(I, S_{i-1}). ----------------
class Qwen2VLExpert:
    name = "qwen2vl"

    def __init__(self):
        self.proc = AutoProcessor.from_pretrained(
            cfg.QWEN2VL_ID, min_pixels=256 * 28 * 28, max_pixels=cfg.QWEN2VL_MAX_PIXELS)
        self.proc.tokenizer.padding_side = "left"
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            cfg.QWEN2VL_ID, torch_dtype=DTYPE)
        self.model.to(DEVICE if DEVICE == "cpu" else "cuda:0").eval()
        _count(self.model, cfg.QWEN2VL_ID)

    @torch.no_grad()
    def describe(self, images, prev_texts):
        texts = []
        for prev in prev_texts:
            # Eq. 2: the expert conditions on BOTH the image and the preceding
            # expert's description S_{i-1}.
            user = ("Describe this image in detail: the objects present, their "
                    "attributes (colour, material, size, state), their spatial "
                    "relations, and the scene. Be specific and factual.")
            if prev:
                user += f"\n\nA previous description said: \"{prev}\"\nExtend and correct it."
            msgs = [{"role": "user",
                     "content": [{"type": "image"}, {"type": "text", "text": user}]}]
            texts.append(self.proc.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True))
        inputs = self.proc(text=texts, images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        out = self.model.generate(**inputs, max_new_tokens=cfg.QWEN2VL_MAX_NEW,
                                  do_sample=False)
        trimmed = [o[len(i):] for i, o in zip(inputs["input_ids"], out)]
        txt = self.proc.batch_decode(trimmed, skip_special_tokens=True)
        # A generation that hits the cap ends mid-sentence; Sec. 3.2 windows are
        # sentences, so a partial tail becomes a junk window.
        return [strip_incomplete_tail(t.strip()) for t in txt]


EXPERT_REGISTRY = {"blip2": (Blip2Expert, cfg.BLIP2_BATCH),
                   "qwen2vl": (Qwen2VLExpert, cfg.QWEN2VL_BATCH)}


def run_coe_cascade(image_names):
    """CoE-based Visual to Language Modeling (Sec. 3.1, Eq. 2-6).

    Experts are loaded one at a time so the cascade fits in 16GB per T4. Each
    expert processes the whole image list before the next expert starts, which
    is mathematically identical to running the chain image-by-image because
    S_i^{(t)} depends only on (I, S_{i-1}^{(t-1)}).
    """
    descriptions = {im: "" for im in image_names}
    for it in range(cfg.COE_ITERATIONS):          # C iterations (Eq. 6)
        for expert_key in cfg.VLM_CHAIN:          # N cascaded experts
            klass, bs = EXPERT_REGISTRY[expert_key]
            print(f"\n[CoE] iteration {it+1}/{cfg.COE_ITERATIONS} expert={expert_key}")
            expert = klass()
            t0 = time.time()
            n_empty_out = 0
            for s in tqdm(range(0, len(image_names), bs), desc=f"{expert_key}"):
                chunk = image_names[s:s + bs]
                imgs = [load_rgb(c) for c in chunk]
                prevs = [descriptions[c] for c in chunk]
                outs = expert.describe(imgs, prevs)
                n_empty_out += sum(1 for o in outs if not o.strip())
                for c, o in zip(chunk, outs):
                    # Eq. 2: S_i^{(t)} replaces S_{i-1}^{(t-1)} as the running
                    # description carried down the chain.
                    descriptions[c] = o if o else descriptions[c]
                if s == 0:
                    per = (time.time() - t0) / len(chunk)
                    print(f"  calibration: {per:.2f}s/image on this expert -> "
                          f"~{per*len(image_names)/60:.1f} min for this stage")
            frac = n_empty_out / max(len(image_names), 1)
            print(f"  {expert_key}: {n_empty_out}/{len(image_names)} "
                  f"({100*frac:.1f}%) empty generations")
            if frac > 0.05:
                print(f"  WARNING: {expert_key} returned nothing for {100*frac:.1f}% of "
                      "images. On a T4 everything runs in fp16 (no bf16 on sm75) and "
                      "Qwen checkpoints are bf16-trained, so this is a plausible "
                      "fp16-overflow signature rather than a prompt problem. "
                      "Check probe_kg_extraction.py's fp16 sanity line.")
            free(expert.model, expert)
    return descriptions


# --- MMKG construction (Sec. 3.3, Eq. 9-10) ---------------------------------
# ASSUMPTION #4: the paper does not give the extraction prompt. This follows the
# delimited-tuple style of LightRAG [27], which Sec. 4.1 names as the framework.
ENT_RE = re.compile(r'\("entity"<\|>(.*?)<\|>(.*?)<\|>(.*?)\)', re.S)
REL_RE = re.compile(r'\("relationship"<\|>(.*?)<\|>(.*?)<\|>(.*?)<\|>(.*?)\)', re.S)

KG_SYSTEM = (
    "You are a knowledge graph extractor. You output only the requested format.")

_KG_SCHEMA_FULL = ('{{"entities": [{{"name": "...", "type": "...", "description": "..."}}],\n'
                   ' "relationships": [{{"source": "...", "target": "...", '
                   '"relation": "...", "description": "..."}}]}}')
_KG_SCHEMA_LEAN = ('{{"entities": [{{"name": "...", "type": "..."}}],\n'
                   ' "relationships": [{{"source": "...", "target": "...", '
                   '"relation": "..."}}]}}')

# ASSUMPTION #4 (revised again): the paper gives no extraction prompt. The lean
# schema drops the "description" field, which retrieval never reads (Eq. 11 matches
# on h/r/t) and which the model left empty for most items in the v2 run. Fewer
# decoded tokens means fewer capped generations and a shorter run.
KG_JSON_TEMPLATE = """Extract the entities and relationships from the text below.

Return one JSON object and nothing else. No markdown fence, no commentary.

%s

Rules:
- "name", "source", "target" are short lowercase noun phrases from the text.
- "relation" is a short lowercase verb or preposition phrase (e.g. "wears", "is a type of", "part of", "has property", "left of").
- "source" and "target" must both be entities, never verbs.
- Include attributes (colour, material, size, state) as relationships.
- Extract only what the text supports. Do not invent facts.

Text:
{text}
""" % (_KG_SCHEMA_FULL if cfg.KG_EMIT_DESCRIPTIONS else _KG_SCHEMA_LEAN)
KG_TEMPLATE = """From the text below, extract every entity and every relationship.

Output format, one item per line, nothing else:
("entity"<|>NAME<|>TYPE<|>SHORT DESCRIPTION)
("relationship"<|>SOURCE ENTITY<|>TARGET ENTITY<|>RELATION<|>SHORT DESCRIPTION)

Rules:
- NAME must be a short lowercase noun phrase that appears in or is implied by the text.
- RELATION must be a short lowercase verb or preposition phrase (e.g. "wears", "is a type of", "part of", "has property", "left of").
- Include attribute relationships (colour, material, size, state) as relationships.
- Extract only what the text supports. Do not invent facts.

Text:
{text}
"""


class LLMRunner:
    """Shared batched HF generation for both the KG stage and the QA stage."""

    def __init__(self, model_id):
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        if DEVICE == "cuda":
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, torch_dtype=DTYPE, device_map="auto",
                max_memory=_max_memory_map())
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, torch_dtype=torch.float32)
        self.model.eval()
        _count(self.model, model_id)
        if hasattr(self.model, "hf_device_map"):
            print("  device map:", set(str(v) for v in self.model.hf_device_map.values()))
        try:
            self.in_dev = next(p.device for p in self.model.parameters()
                               if p.device.type != "meta")
        except StopIteration:
            self.in_dev = torch.device(DEVICE)

    @torch.no_grad()
    def chat(self, system, users, max_new_tokens, batch_size, desc="llm"):
        outs, self.n_capped = [], 0
        for s in tqdm(range(0, len(users), batch_size), desc=desc):
            chunk = users[s:s + batch_size]
            prompts = [self.tok.apply_chat_template(
                [{"role": "system", "content": system},
                 {"role": "user", "content": u}],
                tokenize=False, add_generation_prompt=True) for u in chunk]
            enc = self.tok(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=cfg.LLM_MAX_INPUT_TOKENS)
            enc = {k: v.to(self.in_dev) for k, v in enc.items()}
            gen = self.model.generate(**enc, max_new_tokens=max_new_tokens,
                                      do_sample=False,
                                      pad_token_id=self.tok.pad_token_id)
            trimmed = gen[:, enc["input_ids"].shape[1]:]
            self.n_capped += int((trimmed != self.tok.pad_token_id).sum(dim=1)
                                 .ge(max_new_tokens).sum())
            outs.extend([t.strip() for t in
                         self.tok.batch_decode(trimmed, skip_special_tokens=True)])
        return outs


def _collect(obj_list):
    ents, rels = [], []
    for o in obj_list:
        if not isinstance(o, dict):
            continue
        if {"source", "target", "relation"} <= set(o):
            rels.append((str(o.get("source", "")), str(o.get("target", "")),
                         str(o.get("relation", "")), str(o.get("description", ""))))
        elif "name" in o:
            ents.append((str(o.get("name", "")), str(o.get("type", "")),
                         str(o.get("description", ""))))
    return ents, rels


def _parse_json_graph(raw_text):
    """Tolerant JSON reader.

    v2 ran json.loads on the whole response and returned nothing when it failed.
    353/1200 (+CVs) and 206/1130 (+SV) generations hit the token cap, so the final
    object was truncated and every complete object before it was thrown away with
    it. v3 tries the whole document first, then falls back to salvaging each flat
    {...} object independently: truncation now costs the last item only.
    """
    txt = re.sub(r"^```(?:json)?|```$", "", raw_text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", txt, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            if isinstance(obj, dict) and ("entities" in obj or "relationships" in obj):
                ents, _ = _collect(obj.get("entities") or [])
                _, rels = _collect(obj.get("relationships") or [])
                return ents, rels
        except json.JSONDecodeError as e:
            # Expected on capped generations; salvage below. Not silenced -- the
            # per-tag salvage counts are printed by the caller.
            _ = e
    # Salvage: the objects here are flat, so a non-nested brace match finds them.
    objs = []
    for frag in re.findall(r"\{[^{}]*\}", txt, re.S):
        try:
            objs.append(json.loads(frag))
        except json.JSONDecodeError:
            continue
    return _collect(objs)


def parse_graph(raw_text, image_name):
    """Eq. 10: G = {(h, r, t) | h,t in E, r in R}. Sec. 3.3 also stores the image
    location so entities link back to visual data."""
    entities, triplets = {}, []
    if cfg.KG_FORMAT == "json":
        ent_hits, rel_hits = _parse_json_graph(raw_text)
    elif cfg.KG_FORMAT == "tuple":
        ent_hits = ENT_RE.findall(raw_text)
        rel_hits = REL_RE.findall(raw_text)
    else:
        hard_fail(f"unknown KG_FORMAT {cfg.KG_FORMAT!r}; use 'json' or 'tuple'")
    for name, etype, edesc in ent_hits:
        n = name.strip().lower()
        if n:
            entities[n] = {"type": etype.strip().lower(),
                           "desc": edesc.strip(),
                           "image": os.path.join(cfg.IMAGE_ROOT, image_name)}
    n_dropped = 0
    for h, t, r, d in rel_hits:
        h, t, r = h.strip().lower(), t.strip().lower(), r.strip().lower()
        if cfg.DROP_MALFORMED_TRIPLETS and (h == t or not (h and t and r)):
            n_dropped += 1
            continue
        if h and t and r:
            triplets.append({"h": h, "r": r, "t": t, "desc": d.strip(),
                             "image": os.path.join(cfg.IMAGE_ROOT, image_name)})
    return {"entities": entities, "triplets": triplets, "dropped": n_dropped}


def _kg_template():
    return KG_JSON_TEMPLATE if cfg.KG_FORMAT == "json" else KG_TEMPLATE


def probe_extraction(runner, descriptions, image_names, tag):
    """Extract from a handful of images and stop the run if the yield is bad.

    v1 spent 5.0 GPU-hours on two extraction passes that produced 41 and 217
    triplets over 1200 images. This costs a few minutes and refuses to let that
    happen again.
    """
    probe_imgs = [im for im in image_names if descriptions[im].strip()][:cfg.PROBE_N]
    if not probe_imgs:
        hard_fail(f"[{tag}] every description is empty; nothing to extract from")
    users = [_kg_template().format(text=descriptions[im][:4000]) for im in probe_imgs]
    t0 = time.time()
    raws = runner.chat(KG_SYSTEM, users, cfg.KG_MAX_NEW, cfg.KG_BATCH,
                       desc=f"probe-{tag}")
    per_img = (time.time() - t0) / len(probe_imgs)
    n_ok = 0
    for im, raw in zip(probe_imgs, raws):
        g = parse_graph(raw, im)
        n_ok += int(len(g["triplets"]) > 0)
        print(f"\n--- probe [{tag}] {im}: entities={len(g['entities'])} "
              f"triplets={len(g['triplets'])} raw_chars={len(raw)}")
        print("RAW >>>")
        print(raw[:900] if raw else "(EMPTY STRING)")
        print("<<<")
    rate = n_ok / len(probe_imgs)
    print(f"\n[{tag}] probe parse rate {rate:.2f} "
          f"({n_ok}/{len(probe_imgs)}) with KG_FORMAT={cfg.KG_FORMAT!r}")
    if rate < cfg.PROBE_MIN_PARSE_RATE:
        hard_fail(
            f"[{tag}] extraction parsed {rate:.0%} of probe responses, below "
            f"PROBE_MIN_PARSE_RATE={cfg.PROBE_MIN_PARSE_RATE}. Read the RAW blocks "
            "above: if the model emitted a different shape, switch KG_FORMAT; if it "
            "emitted empty or garbage text, that is fp16 instability on T4 and no "
            "prompt change fixes it. Stopping instead of burning hours on a graph "
            "that will come back empty.")

    proj_h = 2 * per_img * len(image_names) / 3600
    print(f"[{tag}] {per_img:.2f} s/image at KG_BATCH={cfg.KG_BATCH} -> projected "
          f"{proj_h:.2f} h for both extraction passes over {len(image_names)} images")
    if proj_h > cfg.MAX_RUNTIME_HOURS:
        hard_fail(
            f"projected extraction time {proj_h:.2f} h exceeds MAX_RUNTIME_HOURS="
            f"{cfg.MAX_RUNTIME_HOURS}. Lower MAX_IMAGES (any reduction is already a "
            "deviation row), raise KG_BATCH if memory allows, or raise the budget. "
            "Stopping now rather than dying at the 12-hour session limit.")
    return rate, per_img


def build_mmkg(runner, descriptions, image_names, tag):
    """Eq. 9: G = LLM(S_hat (+) T). T is empty here -- see the deviation table.

    Images whose description is empty are SKIPPED, not sent to the LLM. Eq. 9
    builds G from S_hat; with S_hat empty and T empty there is no input, and v1's
    behaviour of calling the model on an empty string made it invent a generic
    graph that then got attached to unrelated images.
    """
    todo = [im for im in image_names
            if descriptions[im].strip() or not cfg.SKIP_EMPTY_SHAT]
    n_skipped = len(image_names) - len(todo)
    print(f"[{tag}] extracting {len(todo)} images, skipping {n_skipped} with empty "
          f"description")
    users = [_kg_template().format(text=descriptions[im][:4000]) for im in todo]
    raws = runner.chat(KG_SYSTEM, users, cfg.KG_MAX_NEW, cfg.KG_BATCH,
                       desc=f"kg-extract-{tag}")
    with open(RAW_KG_LOG, "a") as f:
        for im, raw in zip(todo, raws):
            f.write(json.dumps({"tag": tag, "image": im, "raw": raw}) + "\n")
    kg = {im: {"entities": {}, "triplets": []} for im in image_names}
    n_parsed = 0
    for im, raw in zip(todo, raws):
        g = parse_graph(raw, im)
        kg[im] = g
        n_parsed += int(len(g["triplets"]) > 0)
    print(f"[{tag}] parsed >=1 triplet for {n_parsed}/{len(todo)} extracted images "
          f"({100*n_parsed/max(len(todo),1):.1f}%) | generations that hit the "
          f"{cfg.KG_MAX_NEW}-token cap: {runner.n_capped}")
    return kg


# --- Retrieval (Eq. 11) and prompt augmentation (Eq. 12) -------------------
# ASSUMPTION #3/#4: Sec. 4.1 uses "LightRAG's hybrid retrieval approach"; the
# details live in Appendix D, which is not in the provided text. This is a
# dual-level keyword retriever over the queried image's subgraph: entity-level
# matches (local) are weighted above relation-level matches (global).
STOPWORDS = set("""a an the is are was were be being been do does did of on in at to for
with and or if what which who whom whose this that these those there here it its his her
their your our my as by from into onto over under about you i he she they we can could
would should will shall may might must not no yes any some more most other same""".split())


def query_tokens(q):
    q = q.lower().translate(str.maketrans("", "", string.punctuation))
    return {w for w in q.split() if w and w not in STOPWORDS}


def retrieve(question, subgraph, k):
    qt = query_tokens(question)
    scored = []
    for tr in subgraph["triplets"]:
        ent_tok = set((tr["h"] + " " + tr["t"]).split())
        rel_tok = set(tr["r"].split())
        score = 2 * len(qt & ent_tok) + 1 * len(qt & rel_tok)
        scored.append((score, tr))
    scored.sort(key=lambda x: -x[0])
    # When nothing matches lexically the subgraph is still image-specific, so the
    # first k extracted triplets are used rather than an empty context.
    return [tr for _, tr in scored[:k]]


QA_SYSTEM = ("You answer visual questions about a specific image using the knowledge "
             "graph facts you are given. Reply with the answer only: a single word or "
             "a short noun phrase. No sentence, no explanation, no punctuation.")


def build_prompt(question, triplets):
    """Eq. 12: p_aug = q || ( union over (h,r,t) in G_q of [h]->r->[t] )."""
    if not triplets:
        return f"Question: {question}\nAnswer:"
    facts = "\n".join(f"[{t['h']}]->{t['r']}->[{t['t']}]" for t in triplets)
    return f"Knowledge graph facts about the image:\n{facts}\n\nQuestion: {question}\nAnswer:"


In [ ]:
# %% [CELL 5] ----------------------------------------------------------------
# ===== TRAIN =====
# ----------------------------------------------------------------------------
# There is no training. VaLiK is zero-shot and uses no fine-tuning: Sec. 1 calls
# it "zero-shot", and the Table 1 caption states the evaluation is done "without
# any fine-tuning on the training set". No optimizer, learning rate, schedule,
# epoch count, batch size for training, or regularizer is specified anywhere in
# the paper because none exists.
#
# What this section does instead is the offline construction stage that Eq. 2-10
# describe: cascade the VLMs, prune with CLIP, and extract the graph. Its wall
# clock is what gets reported in the "Train Time" column, and the label is
# explained in the results notes.
TRAINING_STEPS = 0

if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)

# ---- Stage A: CoE cascade (Sec. 3.1) --------------------------------------
t0 = time.time()
descriptions = run_coe_cascade(sel_images)
STAGE_SECONDS["coe_cascade"] = time.time() - t0
with open(CAPTIONS_PATH, "w") as f:
    json.dump({"descriptions": descriptions,
               "seconds": STAGE_SECONDS["coe_cascade"],
               "params": {k: v for k, v in PARAM_COUNTS.items()}}, f)
print(f"CoE cascade: {STAGE_SECONDS['coe_cascade']/60:.1f} min")

empty_desc = sum(1 for im in sel_images if not descriptions[im].strip())
if empty_desc:
    hard_fail(f"{empty_desc} images produced an empty description; the VLM stage "
              "failed and the run cannot continue with blanks.")
desc_lens = [len(descriptions[im].split()) for im in sel_images]
print(f"description length (words): mean {np.mean(desc_lens):.1f} "
      f"min {np.min(desc_lens)} max {np.max(desc_lens)}")

# ---- Stage B: Similarity Verification (Sec. 3.2) --------------------------
t0 = time.time()
clip_model = CLIPModel.from_pretrained(cfg.CLIP_ID, torch_dtype=DTYPE)
clip_model.to(DEVICE if DEVICE == "cpu" else "cuda:0").eval()
clip_proc = CLIPProcessor.from_pretrained(cfg.CLIP_ID)
_count(clip_model, cfg.CLIP_ID)
verifier = SimilarityVerifier(clip_model, clip_proc, cfg.TAU)

pruned, alpha_all, kept_counts, win_counts = {}, [], [], []
for im in tqdm(sel_images, desc="similarity verification"):
    img = load_rgb(im)
    s_hat, kept, alphas = verifier.prune(img, descriptions[im])
    # Eq. 8 can empty a description entirely; the paper does not describe a
    # fallback, so an empty S_hat is kept as empty and reported.
    pruned[im] = s_hat
    alpha_all.extend(alphas)
    kept_counts.append(len(kept))
    win_counts.append(len(alphas))
STAGE_SECONDS["similarity_verification"] = time.time() - t0
free(clip_model, verifier)

a = np.array(alpha_all, dtype=float)
print(f"\nalpha_k over {len(a)} windows: mean {a.mean():.4f} p05 {np.percentile(a,5):.4f} "
      f"p95 {np.percentile(a,95):.4f} min {a.min():.4f} max {a.max():.4f}")
print(f"windows kept at tau={cfg.TAU}: {sum(kept_counts)}/{sum(win_counts)} "
      f"({100*sum(kept_counts)/max(sum(win_counts),1):.1f}%)")
n_empty_sv = sum(1 for im in sel_images if not pruned[im].strip())
print(f"images whose description was fully pruned: {n_empty_sv}")
print(f"SV stage: {STAGE_SECONDS['similarity_verification']/60:.1f} min")
if n_empty_sv:
    print(f"  NOTE: those {n_empty_sv} images get an EMPTY subgraph. Eq. 9 has no "
          "input for them. They are not sent to the extractor.")
with open(PRUNED_PATH, "w") as f:
    json.dump({"pruned": pruned, "alpha_mean": float(a.mean()), "tau": cfg.TAU}, f)

# ---- Stage C: MMKG construction (Sec. 3.3) --------------------------------
kg_runner = LLMRunner(cfg.KG_LLM_ID)

# Cheap gate: verifies the extractor is answering in a parseable shape AND that
# the full passes fit the budget, before either pass starts.
_probe_rate, _probe_per_img = probe_extraction(
    kg_runner, descriptions, sel_images, "probe")

t0 = time.time()
kg_cvs = build_mmkg(kg_runner, descriptions, sel_images, "cvs")
STAGE_SECONDS["kg_cvs"] = time.time() - t0
with open(KG_RAW_PATH, "w") as f:
    json.dump({"kg": kg_cvs, "seconds": STAGE_SECONDS["kg_cvs"]}, f)

t0 = time.time()
kg_sv = build_mmkg(kg_runner, pruned, sel_images, "sv")
STAGE_SECONDS["kg_sv"] = time.time() - t0
with open(KG_SV_PATH, "w") as f:
    json.dump({"kg": kg_sv, "seconds": STAGE_SECONDS["kg_sv"]}, f)


def kg_stats(kg, tag):
    n_ent = len({e for g in kg.values() for e in g["entities"]})
    n_trip = sum(len(g["triplets"]) for g in kg.values())
    n_rel = len({t["r"] for g in kg.values() for t in g["triplets"]})
    empty = sum(1 for g in kg.values() if not g["triplets"])
    print(f"[{tag}] entities {n_ent} | triplets {n_trip} | distinct relations {n_rel} "
          f"| images with no triplet {empty}")
    return {"entities": n_ent, "triplets": n_trip, "relations": n_rel, "empty": empty}


stats_cvs = kg_stats(kg_cvs, "+CVs")
stats_sv = kg_stats(kg_sv, "+SV")
storage_mb = {"+CVs": os.path.getsize(KG_RAW_PATH) / 1e6,
              "+SV": os.path.getsize(KG_SV_PATH) / 1e6}
for tag, st in [("+CVs", stats_cvs), ("+SV", stats_sv)]:
    covered = len(sel_images) - st["empty"]
    if covered < len(sel_images) // 2:
        print(f"  WARNING: {tag} graph covers only {covered}/{len(sel_images)} images. "
              "The KG rows below will differ from the no-KG baseline by noise only "
              "and are NOT a measurement of the method.")
print("MMKG storage (MB):", {k: round(v, 2) for k, v in storage_mb.items()},
      "-- paper reports 489MB for its ScienceQA graph (Sec. 4.2)")

# --- leakage guard, corrected -------------------------------------------------
# v2 asserted that no gold `full_answer` string appears in the graph. That test is
# invalid and it killed a completed run. GQA's full_answer is a short template --
# "the bus is on the street." -- so a captioner that describes the image correctly
# reproduces it by construction. A substring match cannot distinguish reading the
# label from describing the picture, so it flags success as fraud.
#
# What IS a valid structural guarantee: the construction path consumed pixels only.
# run_coe_cascade() takes image_names; SV takes (image, text); build_mmkg() takes a
# description dict keyed by filename. No record field is reachable from any of them.
assert set(descriptions) == set(sel_images), \
    "descriptions are keyed by something other than the selected image files"
assert set(pruned) == set(sel_images), "pruned text is not keyed by image file"
assert set(kg_sv) == set(sel_images) and set(kg_cvs) == set(sel_images), \
    "the graph is keyed by something other than the selected image files"

# And a behavioural check that CAN detect real leakage: `reasoning` and `thought`
# are long, distinctive CoT strings. No captioner coincides with those.
_graph_text = " ".join(t["desc"].lower() for g in kg_sv.values()
                       for t in g["triplets"])
_distinctive = [str(r[f]).strip().lower() for r in eval_records
                for f in ("reasoning", "thought")]
_distinctive = [t for t in _distinctive if len(t) >= 40]
_hits = [t for t in _distinctive if t in _graph_text]
assert not _hits, f"annotation CoT text leaked into the graph: {_hits[:2]}"
print(f"leakage guard: 0 of {len(_distinctive)} distinctive CoT strings appear in "
      "the graph; construction is keyed on image files only.")

# The full_answer overlap is reported, not asserted. A HIGH number here is a good
# sign about description quality, not a bad one about leakage.
_gold = [r["full_answer"].strip().lower() for r in eval_records]
_coincide = sum(1 for g in _gold if g and g in _graph_text)
print(f"diagnostic: {_coincide}/{len(_gold)} gold full_answer template sentences "
      f"({100*_coincide/len(_gold):.1f}%) also occur in the graph text. Expected and "
      "harmless -- these are short GQA templates that any correct description "
      "reproduces. The field is never read as input.")


In [ ]:
# %% [CELL 6] ----------------------------------------------------------------
# ===== EVALUATE =====
# ----------------------------------------------------------------------------
from sklearn.metrics import (multilabel_confusion_matrix,
                             precision_recall_fscore_support)

_PUNCT = str.maketrans("", "", string.punctuation)
_ARTICLES = {"a", "an", "the"}


def normalize_answer(s):
    """ASSUMPTION #8. The paper reports accuracy on multiple-choice / classification
    tasks (Sec. 4.1) and defines no string normalizer, because it never needs one.

    Normalization here: take the first line, drop a leading 'answer:' prefix,
    lowercase, strip all ASCII punctuation, drop the articles a/an/the, and
    collapse runs of whitespace. Nothing else -- no stemming, no synonym table,
    no number-word mapping.
    """
    s = str(s).strip().split("\n")[0]
    s = re.sub(r"^(the\s+)?answer\s*(is)?\s*[:\-]?\s*", "", s, flags=re.I)
    s = s.lower().translate(_PUNCT)
    toks = [w for w in s.split() if w not in _ARTICLES]
    return " ".join(toks).strip()


def run_experiment(runner, name, kg, use_kg):
    prompts, gold = [], []
    for r in eval_records:
        trips = retrieve(r["question"], kg[r["image"]], cfg.TOP_K_TRIPLETS) if use_kg else []
        p = build_prompt(r["question"], trips)
        # Leakage guard on every single prompt.
        for fld in FORBIDDEN_FIELDS:
            v = str(r[fld]).strip()
            if fld != "answer" and len(v) > cfg.LEAK_MIN_CHARS:
                assert v.lower() not in p.lower(), \
                    f"record {r['rid']}: field '{fld}' leaked into the prompt"
        assert r["question"] in p, "prompt lost the query q"
        prompts.append(p)
        gold.append(r["answer"])
    t0 = time.time()
    raw = runner.chat(QA_SYSTEM, prompts, cfg.QA_MAX_NEW, cfg.QA_BATCH, desc=name)
    secs = time.time() - t0
    y_true = [normalize_answer(g) for g in gold]
    y_pred = [normalize_answer(p) for p in raw]
    n_with_facts = sum(1 for p in prompts if p.startswith("Knowledge graph facts"))
    return {"name": name, "raw": raw, "y_true": y_true, "y_pred": y_pred,
            "prompts": prompts, "seconds": secs,
            "frac_prompts_with_facts": n_with_facts / len(prompts),
            "prompt_tokens": float(np.mean([len(p.split()) for p in prompts]))}


def compute_metrics(y_true, y_pred):
    """Every value here comes from y_true/y_pred. No defaults dict, no fallbacks.

    Acc  = normalized exact match. This is the paper's metric family: Sec. 4.1
           says performance "is assessed through accuracy percentage", reported
           here as a fraction in [0,1].
    Prec/Recall/F1 = MACRO averaged, one-vs-rest, over the union of gold and
           predicted answer strings. Micro-averaged P/R/F1 would equal Acc
           exactly in this single-label setting, so macro is the only informative
           choice. The paper does NOT specify an averaging mode; it reports
           accuracy only.
    FPR/FNR = macro one-vs-rest from the multilabel confusion matrix. There is no
           designated positive class -- the label space is an open answer
           vocabulary, so every answer string is treated as its own class.
    ROC-AUC / PR-AUC = N/A. There are no probability scores: the model emits a
           free-form string, not a distribution over a fixed label set. Deriving a
           score from correctness would return 1.0 by construction and measure
           nothing.
    """
    n = len(y_true)
    acc = float(sum(int(t == p) for t, p in zip(y_true, y_pred))) / n
    labels = sorted(set(y_true) | set(y_pred))
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0)
    mcm = multilabel_confusion_matrix(y_true, y_pred, labels=labels)
    tn, fp, fn, tp = mcm[:, 0, 0], mcm[:, 0, 1], mcm[:, 1, 0], mcm[:, 1, 1]
    with np.errstate(divide="ignore", invalid="ignore"):
        fpr_c = np.where((fp + tn) > 0, fp / (fp + tn), np.nan)
        fnr_c = np.where((fn + tp) > 0, fn / (fn + tp), np.nan)
    return {"Acc": acc, "Prec": float(prec), "Recall": float(rec), "F1": float(f1),
            "ROC-AUC": float("nan"), "PR-AUC": float("nan"),
            "FPR": float(np.nanmean(fpr_c)), "FNR": float(np.nanmean(fnr_c)),
            "n_labels": len(labels)}


def flag(name, value):
    # The three literals below are required by the perfect-score check, not
    # used in any computation.
    if isinstance(value, float) and not np.isnan(value) and value in (0.0, 0.5, 1.0):
        print(f"  WARNING: {name} == {value}. This came from a real comparison of "
              f"{len(eval_records)} predictions against ground truth, but an exact "
              f"{value} on an open-vocabulary task is far more likely to be a bug "
              f"(empty predictions, a broken parser, or a collapsed label set) "
              f"than a clean result. Inspect predictions.jsonl before trusting it.")


qa_runner = kg_runner if cfg.LLM_ID == cfg.KG_LLM_ID else LLMRunner(cfg.LLM_ID)

experiments = [("Qwen2.5-7B (no KG)", None, False)]
if cfg.RUN_ABLATION:
    experiments += [("+ CVs (Image-only)", kg_cvs, True),
                    ("+ SV (Image-only, VaLiK)", kg_sv, True)]
else:
    experiments += [("+ SV (Image-only, VaLiK)", kg_sv, True)]

runs, metrics_by_run = [], {}
for name, kg, use_kg in experiments:
    print(f"\n=== {name} ===")
    res = run_experiment(qa_runner, name, kg if kg is not None else {}, use_kg)
    m = compute_metrics(res["y_true"], res["y_pred"])
    for k, v in m.items():
        if isinstance(v, float):
            flag(f"{name} {k}", v)
    n_empty_pred = sum(1 for p in res["y_pred"] if not p)
    print(f"  prompts carrying at least one triplet: "
          f"{100*res['frac_prompts_with_facts']:.1f}% | "
          f"mean prompt words {res['prompt_tokens']:.1f}")
    if use_kg and res["frac_prompts_with_facts"] < 0.5:
        print("  WARNING: most prompts carried no KG evidence, so this row is the "
              "no-KG baseline under a different label. Do not report it as VaLiK.")
    print(f"  n_eval {len(res['y_true'])} | empty predictions {n_empty_pred} | "
          f"distinct predictions {len(set(res['y_pred']))} | "
          f"distinct gold {len(set(res['y_true']))}")
    if n_empty_pred > len(res["y_true"]) // cfg.EMPTY_PRED_WARN_DIV:
        print("  WARNING: >10% of predictions are empty after normalization. "
              "Check QA_MAX_NEW and the answer parser before reading the metrics.")
    res["metrics"] = m
    runs.append(res)
    metrics_by_run[name] = m

# ---- Requested split: with and without full-frame-box records --------------
# GROUNDING ACCURACY IS N/A (no boxes are predicted). This is exact match on the
# same records, split by whether the annotated box covers most of the frame.
ff_mask = np.array([r["is_full_frame"] for r in eval_records])
subset_rows = []
for res in runs:
    yt, yp = np.array(res["y_true"], dtype=object), np.array(res["y_pred"], dtype=object)
    corr = np.array([int(a == b) for a, b in zip(yt, yp)])
    row = {"experiment": res["name"],
           "EM_all": float(corr.mean()),
           "n_all": int(len(corr)),
           "EM_excl_full_frame": float(corr[~ff_mask].mean()) if (~ff_mask).any() else float("nan"),
           "n_excl_full_frame": int((~ff_mask).sum()),
           "EM_full_frame_only": float(corr[ff_mask].mean()) if ff_mask.any() else float("nan"),
           "n_full_frame": int(ff_mask.sum())}
    subset_rows.append(row)
subset_df = pd.DataFrame(subset_rows)
print("\nExact match with and without full-frame-box records "
      "(grounding IoU itself is N/A -- the method predicts no boxes):")
print(subset_df.to_string(index=False))


In [ ]:
# %% [CELL 7] ----------------------------------------------------------------
# ===== SAVE RESULTS =====
# ----------------------------------------------------------------------------
PEAK_GB = peak_gpu_gb()
INPUTS = f"{cfg.EVAL_JSONL} | {cfg.IMAGE_ROOT}"
SOURCE = ("Liu et al., Aligning Vision to Language: Annotation-Free Multimodal "
          "Knowledge Graph Construction for Enhanced LLMs Reasoning (ICCV 2025)")

MODEL_IDS = {
    "Qwen2.5-7B (no KG)": [cfg.LLM_ID],
    "+ CVs (Image-only)": [cfg.BLIP2_ID, cfg.QWEN2VL_ID, cfg.KG_LLM_ID, cfg.LLM_ID],
    "+ SV (Image-only, VaLiK)": [cfg.BLIP2_ID, cfg.QWEN2VL_ID, cfg.CLIP_ID,
                                 cfg.KG_LLM_ID, cfg.LLM_ID],
}
MODEL_NAMES = {
    "Qwen2.5-7B (no KG)": "Qwen2.5-7B-Instruct (7B)",
    "+ CVs (Image-only)": "BLIP2-OPT-2.7B + Qwen2-VL-2B-I -> Qwen2.5-7B-Instruct",
    "+ SV (Image-only, VaLiK)": ("BLIP2-OPT-2.7B + Qwen2-VL-2B-I + CLIP-ViT-L/14 "
                                 "-> Qwen2.5-7B-Instruct"),
}
BUILD_SECONDS = {
    "Qwen2.5-7B (no KG)": float("nan"),   # no offline construction stage exists
    "+ CVs (Image-only)": STAGE_SECONDS["coe_cascade"] + STAGE_SECONDS["kg_cvs"],
    "+ SV (Image-only, VaLiK)": (STAGE_SECONDS["coe_cascade"]
                                 + STAGE_SECONDS["similarity_verification"]
                                 + STAGE_SECONDS["kg_sv"]),
}
KG_STATS = {"Qwen2.5-7B (no KG)": None, "+ CVs (Image-only)": stats_cvs,
            "+ SV (Image-only, VaLiK)": stats_sv}
KG_MB = {"Qwen2.5-7B (no KG)": float("nan"), "+ CVs (Image-only)": storage_mb["+CVs"],
         "+ SV (Image-only, VaLiK)": storage_mb["+SV"]}

rows = []
for res, srow in zip(runs, subset_rows):
    name = res["name"]
    m = res["metrics"]
    ids = MODEL_IDS[name]
    missing_p = [i for i in ids if i not in PARAM_COUNTS]
    if missing_p:
        hard_fail(f"parameter count was never measured for {missing_p}")
    params = int(sum(PARAM_COUNTS[i] for i in set(ids)))
    st = KG_STATS[name]
    rows.append({
        "Acc": m["Acc"], "Prec": m["Prec"], "Recall": m["Recall"], "F1": m["F1"],
        "ROC-AUC": m["ROC-AUC"], "PR-AUC": m["PR-AUC"],
        "FPR": m["FPR"], "FNR": m["FNR"],
        "Train Time": BUILD_SECONDS[name],
        "Params": params,
        "Comm Cost": float("nan"),
        "Training Steps": TRAINING_STEPS,
        "n_eval": len(res["y_true"]),
        "source": SOURCE,
        "split": "GQA visual-CoT val (subset)",
        "model": MODEL_NAMES[name],
        "inputs": INPUTS,
        "fidelity": "adaptation",
        # extras, after the required columns so the 12-column paste still lines up
        "experiment": name,
        "infer_time_s": res["seconds"],
        "peak_gpu_mem_gb": PEAK_GB,
        "tau": cfg.TAU if name != "Qwen2.5-7B (no KG)" else float("nan"),
        "n_images": len(sel_images),
        "kg_entities": st["entities"] if st else float("nan"),
        "kg_triplets": st["triplets"] if st else float("nan"),
        "kg_storage_mb": KG_MB[name],
        "avg_prompt_words": res["prompt_tokens"],
        "pct_prompts_with_facts": 100 * res["frac_prompts_with_facts"],
        "EM_excl_full_frame": srow["EM_excl_full_frame"],
        "n_labels_union": m["n_labels"],
    })

COLS = ["Acc", "Prec", "Recall", "F1", "ROC-AUC", "PR-AUC", "FPR", "FNR",
        "Train Time", "Params", "Comm Cost", "Training Steps",
        "n_eval", "source", "split", "model", "inputs", "fidelity",
        "experiment", "infer_time_s", "peak_gpu_mem_gb", "tau", "n_images",
        "kg_entities", "kg_triplets", "kg_storage_mb", "avg_prompt_words",
        "pct_prompts_with_facts", "EM_excl_full_frame", "n_labels_union"]
results = pd.DataFrame(rows)[COLS]

# nan lives inside the code; it becomes the literal text N/A only on the way out.
out = results.copy()
for c in out.columns:
    out[c] = out[c].map(lambda v: "N/A" if isinstance(v, float) and np.isnan(v) else v)

pd.set_option("display.width", 250, "display.max_columns", 60)
print("\n================ RESULTS ================")
print(out.to_string(index=False))
print("=========================================")
print("""
Column notes:
  Acc          normalized exact match, fraction in [0,1]. Sec. 4.1 reports
               accuracy as a percentage; multiply by 100 for the paper's units.
  Prec/Rec/F1  MACRO, one-vs-rest, over the union of gold and predicted answer
               strings. No positive class is designated -- the label space is an
               open answer vocabulary. The paper specifies neither an averaging
               mode nor a positive class; it reports accuracy only.
  ROC-AUC      N/A: no probability score exists. The model emits a free-form
  PR-AUC       string, not a distribution over a fixed label set.
  FPR/FNR      macro one-vs-rest over the same open label set.
  Train Time   NOT training time -- nothing is trained (zero-shot, Sec. 1). This
               is the measured wall clock of the offline construction stage
               (CoE cascade + similarity verification + graph extraction).
               N/A for the no-KG baseline, which has no such stage.
  Comm Cost    N/A: the paper measures storage (489MB, Sec. 4.2), not
               communication. Measured storage is in kg_storage_mb.
  Training Steps 0: inference only.
  fidelity     'adaptation': the paper evaluates CrisisMMD and ScienceQA
               (Sec. 4.1), not GQA. No cell here is comparable to Tables 1-5.
""")

results.to_csv(os.path.join(cfg.OUT_DIR, "results_raw_nan.csv"), index=False)
out.to_csv(os.path.join(cfg.OUT_DIR, "results.csv"), index=False)
subset_df.to_csv(os.path.join(cfg.OUT_DIR, "em_by_box_subset.csv"), index=False)

with open(os.path.join(cfg.OUT_DIR, "predictions.jsonl"), "w") as f:
    for res in runs:
        for r, raw, yt, yp, pr in zip(eval_records, res["raw"], res["y_true"],
                                      res["y_pred"], res["prompts"]):
            f.write(json.dumps({
                "experiment": res["name"], "rid": r["rid"], "image": r["image"],
                "question": r["question"], "gold": r["answer"],
                "gold_norm": yt, "raw_prediction": raw, "pred_norm": yp,
                "correct": int(yt == yp), "is_full_frame": bool(r["is_full_frame"]),
                "prompt": pr}) + "\n")

with open(os.path.join(cfg.OUT_DIR, "run_manifest.json"), "w") as f:
    json.dump({"config": {k: str(v) for k, v in asdict(cfg).items()},
               "stage_seconds": STAGE_SECONDS,
               "param_counts": PARAM_COUNTS,
               "peak_gpu_mem_gb": PEAK_GB,
               "n_images": len(sel_images), "n_eval": len(eval_records),
               "n_records_after_size_drop": len(records),
               "n_boxes_clamped": n_clamped,
               "n_full_frame_records": int(ff_mask.sum()),
               "alpha_mean": float(a.mean()), "tau": cfg.TAU,
               "probe_parse_rate": _probe_rate,
               "probe_seconds_per_image": _probe_per_img,
               "kg_format": cfg.KG_FORMAT,
               "kg_emit_descriptions": cfg.KG_EMIT_DESCRIPTIONS,
               "drop_malformed_triplets": cfg.DROP_MALFORMED_TRIPLETS,
               "standalone": True,
               "kg_stats": {"cvs": stats_cvs, "sv": stats_sv},
               "torch": torch.__version__, "seed": cfg.SEED}, f, indent=2)

print("wrote:")
for fn in ["results.csv", "results_raw_nan.csv", "em_by_box_subset.csv",
           "predictions.jsonl", "run_manifest.json", "captions.json",
           "kg_cvs.json", "kg_sv.json"]:
    p = os.path.join(cfg.OUT_DIR, fn)
    if os.path.exists(p):
        print(f"  {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
